[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_16_masked_diffusion_solution.ipynb)

# 🟡 Solution: Masked Diffusion LM Loss

*Training · Medium*

Reference implementation. Try it yourself in `b_16_masked_diffusion.ipynb` first.

---
A masked diffusion LM is trained by corrupting a sequence to `[MASK]` and asking
the model to fill it back in. The forward process is per-token and independent:
with the linear schedule $\alpha_t = 1 - t$, each position survives with
probability $\alpha_t$ and becomes `[MASK]` with probability $t$.

The continuous-time NELBO collapses into something you already know — a
cross-entropy, weighted by $1/t$ and restricted to the masked positions:

$$\mathcal{L} = \mathbb{E}_{t \sim U(0,1]}\;
\frac{1}{t}\,\frac{1}{L}\sum_{i=1}^{L}
\mathbb{1}[x_t^i = \texttt{[MASK]}]\;\bigl(-\log p_\theta(x_0^i \mid x_t)\bigr)$$

Implement both halves: the forward corruption, and the loss for one $(t, x_t)$
sample.

> Want the mechanism before the objective? Skim **b_17**'s problem statement
> first — it is the reverse process this loss trains, and watching an
> all-`[MASK]` sequence fill in makes the $1/t$ weighting below easier to see.
> Come back here to implement, though: `q_sample` is the forward kernel b_17's
> posterior is derived from.

### Rules
- `q_sample(key, x0, t, *, mask_id)` -> `x_t`, same shape as `x0`
- `masked_diffusion_loss(logits, x0, xt, t, *, mask_id)` -> **scalar**
- `logits` is `(B, L, V)`, `x0` and `xt` are `(B, L)` integer ids, `t` is `(B,)`
  with $t \in (0, 1]$ — one time per sequence, not one per token
- Each position is masked **independently**; `q_sample` must not mask a fixed
  count per sequence
- The model may never predict `[MASK]`: drive that column to $-\infty$ **before**
  normalising, so it holds no probability mass at all
- Positions that were not masked contribute **exactly zero**
- `jax.nn.log_softmax` and `logsumexp` are allowed here — you wrote both in 16
  and b_14, and the point of this problem is elsewhere. `optax` is still banned
- Must work under `jit` and `vmap`

### The denominator is the whole problem
Divide by the number of **masked** tokens and it will look right, train, and be
wrong. The $1/t$ factor already accounts for how many tokens the forward process
masks: $\mathbb{E}[\#\text{masked}] = tL$, so $\frac{1}{t}\sum_{\text{masked}}$
is an unbiased estimator of the full NELBO. Dividing by the *realised* count
corrects for the same thing a second time, and the estimator stops being the
bound you meant to optimise — the gradient is now systematically reweighted
towards the high-noise end.

Divide by $L$ instead, and what you get is the bound in per-token units, which
is exactly what a bits-per-token number needs.

There is a cheap way to remember which is which: take a batch where every
sequence sees the same $t$ but happens to get a different number of masked
tokens. The loss **should** differ between them — more masked tokens is more
evidence, and averaging that away throws it out.

### Why the MASK column must be $-\infty$
`[MASK]` is in the vocabulary, so an unconstrained softmax will happily put mass
on it — mass that can only be wrong, since $x_0$ never contains `[MASK]`. Worse,
it is the easy answer: predicting the token that is literally sitting in the
input is the shortcut every underfit model finds first.

Zeroing it *after* the softmax is not the same thing — the remaining
probabilities no longer sum to one, so the loss is no longer a log-likelihood.
Setting the logit to $-\infty$ before normalising redistributes that mass over
the real vocabulary, which is what the SUBS ("zero masking probability")
parameterisation means in the MDLM paper.

### Absorbing vs uniform, and where flow matching fits
The other classic discrete kernel is **uniform** (D3PM-uniform, SEDD-uniform):
instead of an absorbing `[MASK]`, a corrupted token jumps to a *random* token.
That kernel has no absorbing state, so its posterior needs the full transition
matrix $Q = \alpha I + \frac{1-\alpha}{V}\mathbf{1}\mathbf{1}^\top$ and the loss
becomes a KL between categoricals. Absorbing wins in practice for a structural
reason worth being able to say out loud: `[MASK]` is *self-identifying*. The
model can see exactly which positions are corrupted, already-decoded tokens are
never destroyed again, and the posterior collapses to "stay masked, or jump
straight to $x_0$" — no matrix products anywhere.

**Discrete flow matching** (Campbell et al., Gat et al. 2024) with a masked
source distribution and a linear $\kappa$ schedule produces *this same
objective*, term for term. The framings differ, and the sampler you get out of
the flow view is more general (it can un-decode a token, which the diffusion
posterior above cannot), but if you are asked "how is discrete flow matching
different from masked diffusion", the honest answer starts with "the training
loss is the same". MeanFlow is a different animal again: average-velocity fields
in *continuous* space for one-step generation, not a discrete-token method.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def q_sample(key, x0, t, *, mask_id):
    x0 = jnp.asarray(x0)
    t = jnp.asarray(t)
    # One draw PER POSITION, compared against this sequence's t: independent
    # masking, so the count varies from sequence to sequence.
    u = jax.random.uniform(key, x0.shape)
    return jnp.where(u < t[..., None], mask_id, x0)


def masked_diffusion_loss(logits, x0, xt, t, *, mask_id):
    logits = jnp.asarray(logits)
    x0 = jnp.asarray(x0)
    t = jnp.asarray(t)

    # The model may never predict [MASK]. Killing the logit BEFORE the
    # normalisation is what redistributes that mass over the real vocabulary.
    logits = logits.at[..., mask_id].set(-jnp.inf)
    log_probs = jax.nn.log_softmax(logits, axis=-1)

    nll = -jnp.take_along_axis(log_probs, x0[..., None], axis=-1)[..., 0]

    # Only the masked positions carry loss; the rest contribute exactly zero.
    masked = xt == mask_id
    per_token = jnp.where(masked, nll, 0.0)

    # Divide by the sequence LENGTH, not by the number of masked tokens: the
    # 1/t factor already accounts for E[#masked] = t * L.
    per_seq = jnp.sum(per_token, axis=-1) / x0.shape[-1]
    return jnp.mean(per_seq / t)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

V, MASK = 8, 7                      # ids 0..6 are real tokens, 7 is [MASK]
x0 = jnp.array([[0, 1, 2, 3, 4, 5]])
t = jnp.array([0.5])

xt = q_sample(jax.random.key(0), x0, t, mask_id=MASK)
print("x0:", x0[0], "\nxt:", xt[0], " masked:", int(jnp.sum(xt == MASK)), "of 6")

# Independent masking: the count varies run to run, it is not exactly t * L.
counts = [int(jnp.sum(q_sample(jax.random.key(k), x0, t, mask_id=MASK) == MASK))
          for k in range(8)]
print("counts over 8 keys:", counts)

# A perfect model scores zero loss.
perfect = jax.nn.one_hot(x0, V) * 100.0
print("\nperfect model:", float(masked_diffusion_loss(perfect, x0, xt, t, mask_id=MASK)))

# A uniform model over the 7 REAL tokens: each masked position costs log(7).
flat = jnp.zeros((1, 6, V))
n_masked = int(jnp.sum(xt == MASK))
print("uniform model:", float(masked_diffusion_loss(flat, x0, xt, t, mask_id=MASK)))
print("by hand      :", float((1 / t[0]) * n_masked / 6 * jnp.log(V - 1)))

# The MASK column is inert: poisoning it changes nothing.
poisoned = flat.at[..., MASK].set(50.0)
print("\npoisoned MASK logit:", float(masked_diffusion_loss(poisoned, x0, xt, t, mask_id=MASK)))

# Same t, more masked tokens -> MORE loss. Dividing by #masked would flatten
# this to a constant, which is the bug this problem is about.
for k in (2, 5):
    xk = q_sample(jax.random.key(k), x0, t, mask_id=MASK)
    print(f"  {int(jnp.sum(xk == MASK))} masked -> "
          f"{float(masked_diffusion_loss(flat, x0, xk, t, mask_id=MASK)):.4f}")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("masked_diffusion")